In [1]:
import torch
import torch.nn as nn

In [50]:
from dataclasses import dataclass

@dataclass
class ModelArgs:
    n_dim: int
    n_experts: int
    n_shared_experts: int # 增加一个共享专家数
    top_k: int
    dropout: float
    batch: int
    seq_len: int

    #新增负载均衡的参数
    bias_lr: float=0.001 #增加一个bias增量参数
    enable_bias: bool = False
    enable_seq_aux: bool = False
    alpha: float = 0.1





In [51]:
class TopkRouter(nn.Module):
    """定义一个topk MOE门控单元"""
    def __init__(self, args):
        super(TopkRouter, self).__init__()
        self.top_k = args.top_k

        # top-k的学习层
        self.topk_linear = nn.Linear(args.n_dim, args.n_experts)

        # 定义噪声可学习参数
        self.noise_linear = nn.Linear(args.n_dim, args.n_experts)

    def forward(self, mha_ouput,bias):
        # logits
        logits = self.topk_linear(mha_ouput)

        ## 路由 bias
        logits = logits + bias

        # top-k
        top_k_logits, indices = logits.topk(self.top_k, dim=-1)

        infs = torch.full_like(logits, float('-inf'))
        sparse_logits = infs.scatter(-1, indices, top_k_logits)
        router_output = nn.functional.softmax(sparse_logits, dim=-1)

        return router_output, indices

In [52]:
class DeepSeekExpert(nn.Module):
    def __init__(self, args, bias=True):
        super().__init__()
        self.gate_proj = nn.Linear(args.n_dim, 4*args.n_dim, bias=bias)
        self.up_proj = nn.Linear(args.n_dim, 4*args.n_dim, bias=bias)
        self.down_proj = nn.Linear(4*args.n_dim, args.n_dim, bias=bias)

    def forward(self, x):
        gate = self.gate_proj(x)
        up = self.up_proj(x)
        return self.down_proj(nn.functional.silu(gate) * up)

In [53]:
class DeepSeekMoE(nn.Module):
    def __init__(self, args):
        super(DeepSeekMoE, self).__init__()
        self.router = TopkRouter(args)
        self.routed_experts = nn.ModuleList([DeepSeekExpert(args) for _ in range(args.n_experts)])
        self.shared_experts = nn.ModuleList([DeepSeekExpert(args) for _ in range(args.n_shared_experts)])
        self.top_k = args.top_k
        self.n_experts = args.n_experts

        # 2.Auxiliary-loss-free load balancing
        self.register_buffer('expert_bias', torch.zeros(self.n_experts))
        self.bias_update_rate = args.bias_lr
        self.enable_bias = args.enable_bias

        # 1.Auxiliary-loss seq load balancing
        self.enable_seq_aux = args.enable_seq_aux
        self.alpha = args.alpha


    def forward(self, x):

        ### batch size
        bsz, seq_len, _ = x.shape

        ## 门控路由输出
        gating_output, indices = self.router(x, self.expert_bias)

        # Reshape可以做batch处理
        flat_x = x.view(-1, x.size(-1))
        flat_gating_output = gating_output.view(-1, gating_output.size(-1))
        print("selected experts: ", gating_output)

        ## 计算专家输出
        final_output = torch.zeros_like(x)
        expert_usage = torch.zeros(self.n_experts)

        # 并行处理每一个routed专家
        for i, expert in enumerate(self.routed_experts):
            expert_mask = (indices == i).any(dim=-1)
            flat_mask = expert_mask.view(-1)

            ## 更新专家的利用次数
            expert_usage[i] = expert_mask.sum().float()

            # 如果专家被选中了
            if flat_mask.any():
                expert_input = flat_x[flat_mask]
                expert_output = expert(expert_input)

                # 专家输出加权求和
                gating_scores = flat_gating_output[flat_mask, i].unsqueeze(1)
                weighted_output = expert_output * gating_scores

                # 更新输出
                final_output[expert_mask] += weighted_output.squeeze(1)

        # 并行处理每一个shared专家
        for i, expert in enumerate(self.shared_experts):
            # bsz > 1时需要reshape到[B, L, d]
            final_output += expert(flat_x).reshape([bsz, seq_len, -1])

        ## Auxiliary-loss-free load balancing
        if self.enable_bias:
            avg_usage = expert_usage.mean()
            for i in range(self.n_experts):
                # 如果该专家大于所有专家平均的负载，就减少其路由概率
                if expert_usage[i] > avg_usage:
                    self.expert_bias[i] -= self.bias_update_rate
                else:
                    self.expert_bias[i] += self.bias_update_rate

        ### Auxiliary-loss seq load balancing
        aux_loss = None
        topk_idx_for_aux_loss = indices.view(bsz, seq_len*self.top_k)
        if self.enable_seq_aux:
            """
            aux_loss = alpha * sum(fi*pi)
            fi = N'/(K'*T) * (序列token选择了第i个专家的数量)
            pi = mean(s(i,t)), s(i,t)是第t个token对第i个专家的亲和度，pi也就是序列token对专家i的平均亲和度, t=1~T
            N‘：路由专家总数
            K'：激活的专家数
            T: 序列长度
            """
            scores_for_seq_aux = gating_output.view(bsz, seq_len, self.n_experts)
            print(1, scores_for_seq_aux.shape)
            fi = torch.zeros(bsz, self.n_experts)
            fi = torch.scatter_add(fi, 1, topk_idx_for_aux_loss, torch.ones(bsz, seq_len * self.top_k))
            print(2, topk_idx_for_aux_loss)
            print(3, fi)
            # [bsz, self.n_experts]
            fi = torch.div(fi, seq_len * self.top_k / self.n_experts)
            # [bsz, self.n_experts]
            pi = scores_for_seq_aux.mean(dim = 1)
            print("top-k index:", indices)
            print("chosed experts tokens:", fi)
            print("fi:", fi)
            print("pi:", pi)
            # [bsz, 1]
            aux_loss = (fi * pi).sum(dim = 1).mean() * self.alpha

        return final_output, expert_usage, aux_loss

In [55]:
torch.manual_seed(2)
args = ModelArgs(n_dim=32, n_experts=4, n_shared_experts=1,
                 top_k=2, dropout=0.1, batch=1, seq_len=5,
                 bias_lr=0.1, enable_seq_aux=True)

deepseek_moe = DeepSeekMoE(args)
for i in range(5):
    mha_output = torch.randn(args.batch, args.seq_len, args.n_dim)
    final_output, avg_usage, aux_loss = deepseek_moe(mha_output)
    print("aux loss:", aux_loss)
    print("output shape:", final_output.shape)
    print("="*100)

selected experts:  tensor([[[0.0000, 0.4505, 0.5495, 0.0000],
         [0.0000, 0.5151, 0.0000, 0.4849],
         [0.5567, 0.0000, 0.0000, 0.4433],
         [0.0000, 0.0000, 0.6166, 0.3834],
         [0.3930, 0.0000, 0.0000, 0.6070]]], grad_fn=<SoftmaxBackward0>)
1 torch.Size([1, 5, 4])
2 tensor([[2, 1, 1, 3, 0, 3, 2, 3, 3, 0]])
3 tensor([[2., 2., 2., 4.]])
top-k index: tensor([[[2, 1],
         [1, 3],
         [0, 3],
         [2, 3],
         [3, 0]]])
chosed experts tokens: tensor([[0.8000, 0.8000, 0.8000, 1.6000]])
fi: tensor([[0.8000, 0.8000, 0.8000, 1.6000]])
pi: tensor([[0.1899, 0.1931, 0.2332, 0.3837]], grad_fn=<MeanBackward1>)
aux loss: tensor(0.1107, grad_fn=<MulBackward0>)
output shape: torch.Size([1, 5, 32])
selected experts:  tensor([[[0.4016, 0.0000, 0.0000, 0.5984],
         [0.5890, 0.0000, 0.4110, 0.0000],
         [0.0000, 0.5114, 0.4886, 0.0000],
         [0.0000, 0.0000, 0.5429, 0.4571],
         [0.0000, 0.4956, 0.5044, 0.0000]]], grad_fn=<SoftmaxBackward0>)
1 tor

In [56]:
torch.manual_seed(2)
args = ModelArgs(n_dim=32, n_experts=4, n_shared_experts=1,
                 top_k=2, dropout=0.1, batch=1, seq_len=5,
                 bias_lr=0.1, enable_bias=True)
mha_output = torch.randn(args.batch, args.seq_len, args.n_dim)

deepseek_moe = DeepSeekMoE(args)
for i in range(5):
    final_output, avg_usage, _ = deepseek_moe(mha_output)
    print("expert avg_usage:", avg_usage)
    print("expert bias:", deepseek_moe.expert_bias)
    print("output shape:", final_output.shape)
    print("="*100)

selected experts:  tensor([[[0.6971, 0.3029, 0.0000, 0.0000],
         [0.5361, 0.0000, 0.4639, 0.0000],
         [0.4615, 0.5385, 0.0000, 0.0000],
         [0.5467, 0.0000, 0.0000, 0.4533],
         [0.6890, 0.3110, 0.0000, 0.0000]]], grad_fn=<SoftmaxBackward0>)
expert avg_usage: tensor([5., 3., 1., 1.])
expert bias: tensor([-0.1000, -0.1000,  0.1000,  0.1000])
output shape: torch.Size([1, 5, 32])
selected experts:  tensor([[[0.6971, 0.3029, 0.0000, 0.0000],
         [0.4862, 0.0000, 0.5138, 0.0000],
         [0.0000, 0.5365, 0.4635, 0.0000],
         [0.4968, 0.0000, 0.0000, 0.5032],
         [0.6890, 0.3110, 0.0000, 0.0000]]], grad_fn=<SoftmaxBackward0>)
expert avg_usage: tensor([4., 3., 2., 1.])
expert bias: tensor([-0.2000, -0.2000,  0.2000,  0.2000])
output shape: torch.Size([1, 5, 32])
selected experts:  tensor([[[0.6971, 0.3029, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.5359, 0.4641],
         [0.0000, 0.0000, 0.5069, 0.4931],
         [0.4470, 0.0000, 0.0000, 0.5530],
     